# GEM_01: Data Loading
## Creates an input file (gem_input_data.pkl) with all the modalities to input into the GEM

## 1: GEM Data Loading and Preparation

In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import os
import pickle
sc.settings.verbosity = 0

In [3]:
import pandas as pd
import numpy as np
import scanpy as sc
import pickle, os

pkgs = {"pandas": pd, "numpy": np, "scanpy": sc}
for name, mod in pkgs.items():
    print(f"{name}: {mod.__version__}")

import sys
print(sys.version)

pandas: 2.3.3
numpy: 2.0.2
scanpy: 1.10.3
3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 18:02:02) 
[Clang 18.1.8 ]


In [4]:
import importlib.metadata

jupyter_pkgs = ["notebook", "jupyterlab", "ipython", "ipykernel", "nbformat", "nbconvert"]
for p in jupyter_pkgs:
    try:
        print(f"{p}: {importlib.metadata.version(p)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{p}: not installed")

notebook: 7.5.5
jupyterlab: 4.5.6
ipython: 8.18.1
ipykernel: 6.31.0
nbformat: 5.10.4
nbconvert: 7.17.1


In [2]:
BASE     = "/Users/maisievarcoe/Desktop/Research_Project/Scripts/Integration/GEM/pyglpk/Data"
 
PATHS = {
    # Tier 1 data (from R)
    "master_table":  f"{BASE}/master_table_final.csv",
    "sc_bridge":     f"{BASE}/scrna_bridge.csv",
    "conf_metabs":   f"{BASE}/high_conf_metabolites.csv",
    "conf_paths":    f"{BASE}/high_conf_pathways.csv",
    "nmr":           f"{BASE}/nmr_matched.csv",
    "microbiome":    f"{BASE}/species_matched.csv",
 
    # Single-cell expression (from R export)
    "mean_expr":     f"{BASE}/mean_expr_per_celltype.csv",
    "mean_expr_high":f"{BASE}/mean_expr_high_sa.csv",
    "mean_expr_low": f"{BASE}/mean_expr_low_sa.csv",
    "mean_expr_ker1_by_sample": f"{BASE}/mean_expr_KER1_by_sample.csv",
    "adata":         f"{BASE}/adata_qc.h5ad",
    "mean_expr_tcell_by_sample": f"{BASE}/mean_expr_TCELL_by_sample.csv",
    "cell_counts_ker1"          : f"{BASE}/cell_counts_KER1.csv",
    "cell_counts_tcell"         : f"{BASE}/cell_counts_TCELL.csv"
}

 
# Availability check
print("File availability:")
for name, path in PATHS.items():
    status = "✓" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"  {status}  {name}")

File availability:
  ✓  master_table
  ✓  sc_bridge
  ✓  conf_metabs
  ✓  conf_paths
  ✓  nmr
  ✓  microbiome
  ✓  mean_expr
  ✓  mean_expr_high
  ✓  mean_expr_low
  ✓  mean_expr_ker1_by_sample
  ✓  adata
  ✓  mean_expr_tcell_by_sample
  ✓  cell_counts_ker1
  ✓  cell_counts_tcell


### 1.1: Load Tier 1 Data

In [3]:

# 1. Load the master table
master_table = pd.read_csv(PATHS["master_table"])
sc_bridge = pd.read_csv(PATHS["sc_bridge"])

# 2. Define your metadata columns exactly as they appear in your list
meta_cols = [
    'NMR_sample_id', 'eg_id', 'sample_short', 'sample_id', 'sa_abundance', 
    'sample_number', 'sample_site', 'lesional', 'local_EASI_intensity', 
    's_aureus_culture_growth_viapath', 'age', 'sex', 'EASI', 
    'weeks_since_last_systemic', 'weeks_since_last_TCS_or_TCI', 
    'days_since_any_topicals', 'TEWL_mean', 'pH_mean', 'SA_status'
]

# 3. Separate metadata and pathways
# We set 'NMR_sample_id' as the index here
metadata = master_table[meta_cols].set_index('NMR_sample_id').copy()

# The pathways are everything else
pathway_cols = [c for c in master_table.columns if c not in meta_cols]
pathways_from_master = master_table[['NMR_sample_id'] + pathway_cols].set_index('NMR_sample_id').copy()

# 4. Clean indices (to ensure no whitespace/hidden chars)
def clean_index(df):
    df.index = df.index.astype(str).str.strip().str.replace(r'[\r\n]', '', regex=True)
    return df

metadata = clean_index(metadata)
pathways_from_master = clean_index(pathways_from_master)

# 5. Now filter using the bridge
valid_eg_ids = set(sc_bridge['eg_id'].astype(str).str.strip())
metadata_filtered = metadata[metadata['eg_id'].astype(str).str.strip().isin(valid_eg_ids)].copy()

# 1. Load data and force the sample ID column to be the index
nmr = pd.read_csv(PATHS["nmr"], index_col=0) # Use the first column as the index
microbiome = pd.read_csv(PATHS["microbiome"], index_col=0)

# 2. Force the index to be strings and strip whitespace to match your metadata
nmr.index = nmr.index.astype(str).str.strip()
microbiome.index = microbiome.index.astype(str).str.strip()

# 3. Quick validation check
print(f"Are all 9 filtered samples in NMR index? {metadata_filtered.index.isin(nmr.index).all()}")

print(f"Filtered metadata shape: {metadata_filtered.shape}")

# Remove the 'nan' row if it exists
if 'nan' in metadata_filtered.index:
    metadata_filtered = metadata_filtered.drop('nan', errors='ignore')
    print("Dropped 'nan' from metadata index.")

Are all 9 filtered samples in NMR index? False
Filtered metadata shape: (9, 18)
Dropped 'nan' from metadata index.


### 1.2: Load Single-Cell Cell-Type Expression Data

In [4]:
mean_expr      = pd.read_csv(PATHS["mean_expr"],      index_col=0)
mean_expr_high = pd.read_csv(PATHS["mean_expr_high"], index_col=0)
mean_expr_low  = pd.read_csv(PATHS["mean_expr_low"],  index_col=0)
mean_expr_ker1_by_sample  = pd.read_csv(PATHS["mean_expr_ker1_by_sample"],  index_col=0)
mean_expr_tcell_by_sample = pd.read_csv(PATHS["mean_expr_tcell_by_sample"], index_col=0)
cell_counts_ker1  = pd.read_csv(PATHS["cell_counts_ker1"],  index_col="sample")
cell_counts_tcell = pd.read_csv(PATHS["cell_counts_tcell"], index_col="sample")
conf_paths  = pd.read_csv(PATHS["conf_paths"])
conf_metabs = pd.read_csv(PATHS["conf_metabs"])

print(f"mean_expr:           {mean_expr.shape}   (genes × cell types)")
print(f"mean_expr_high:      {mean_expr_high.shape}")
print(f"mean_expr_low:       {mean_expr_low.shape}")
print(f"Cell types: {list(mean_expr.columns)}")
print(f"mean_expr_ker1_by_sample:  {mean_expr_ker1_by_sample.shape}  (expect 23017 × 6)")
print(f"mean_expr_tcell_by_sample: {mean_expr_tcell_by_sample.shape}  (expect 23017 × 6)")
print(cell_counts_ker1)
print(cell_counts_tcell)

mean_expr:           (23017, 11)   (genes × cell types)
mean_expr_high:      (23017, 11)
mean_expr_low:       (23017, 11)
Cell types: ['IFE SB', 'OB', 'IFE B', 'KER2', 'T CELL', 'IFE C', 'KER1', 'MEL', 'B CELL', 'uHF SB', 'FIB']
mean_expr_ker1_by_sample:  (23017, 6)  (expect 23017 × 6)
mean_expr_tcell_by_sample: (23017, 6)  (expect 23017 × 6)
                       n_cells
sample                        
GYECZ1191.ABDO.LES          85
GYECZ1191.ABDO.NONLES       54
GYECZ1193.LACF.LES         177
GYECZ1193.LACF.NONLES      253
GYECZ1197.RACF.NONLES       59
GYECZ1197.BACK.LES         141
                       n_cells
sample                        
GYECZ1191.ABDO.LES         375
GYECZ1191.ABDO.NONLES      223
GYECZ1193.LACF.LES         165
GYECZ1193.LACF.NONLES        8
GYECZ1197.RACF.NONLES      114
GYECZ1197.BACK.LES         599


### 1.3: Handle Metadata and Pathways

In [5]:
# 1. Initialization and Cleaning
def clean_index(df):
    df.index = df.index.astype(str).str.strip().str.replace("\r", "", regex=False).str.replace("\n", "", regex=False)
    return df

# Load your four main dataframes
metadata = clean_index(master_table[meta_cols].dropna(subset=["NMR_sample_id"]).set_index("NMR_sample_id"))
pathways = clean_index(master_table[pathway_cols].copy())
pathways.index = master_table["NMR_sample_id"].astype(str)
nmr = clean_index(nmr)
microbiome = clean_index(microbiome)

# 2. Filter by scRNA-seq bridge
valid_eg_ids = set(sc_bridge['eg_id'].astype(str).str.strip())
metadata_filtered = metadata[metadata['eg_id'].isin(valid_eg_ids)].copy()

# 1. First, define the 'common_ids' based on the intersection of ALL your data sources
# This ensures that a sample ID exists in your metadata, NMR data, microbiome data, AND pathways
common_ids_all = sorted(
    set(metadata_filtered.index) & 
    set(pathways.index) & 
    set(nmr.index) & 
    set(microbiome.index)
)

# 2. Now, apply your strict manual filter (the 'gatekeeper')
sc_sample_list = [
    'GYECZ1191.ABDO.LES', 
    'GYECZ1191.ABDO.NONLES', 
    'GYECZ1193.LACF.LES', 
    'GYECZ1193.LACF.NONLES', 
    'GYECZ1197.RACF.NONLES', 
    'GYECZ1197.BACK.LES'
]

# Create the final list by intersecting the "all available" IDs with your "target" list
common_ids = sorted(
    set(common_ids_all) & set(sc_sample_list)
)

# 3. Apply the final subsetting to all objects
metadata = metadata.loc[common_ids]
pathways_from_master = pathways_from_master.loc[common_ids]
#pathways = pathways.loc[common_ids]
nmr = nmr.loc[common_ids]
microbiome = microbiome.loc[common_ids]

print(f"Final sample count after filtering: {len(common_ids)}")

for name, df in {"KER1 expr": mean_expr_ker1_by_sample, "T CELL expr": mean_expr_tcell_by_sample}.items():
    print(f"{name}: columns match common_ids: {set(df.columns) == set(common_ids)}")

for name, df in {"KER1 counts": cell_counts_ker1, "T CELL counts": cell_counts_tcell}.items():
    print(f"{name}: index matches common_ids: {set(df.index) == set(common_ids)}")


Final sample count after filtering: 6
KER1 expr: columns match common_ids: True
T CELL expr: columns match common_ids: True
KER1 counts: index matches common_ids: True
T CELL counts: index matches common_ids: True


### 2.1: Assign S. aureus Status (Low <20%, High >50%, exclude 20-50%)

In [6]:
# 1. ALWAYS filter the metadata to the common_ids first
metadata = metadata.loc[common_ids]

# 2. THEN apply the status assignment
def assign_sa_status(sa):
    if pd.isna(sa):   return None
    elif sa < 20:     return 'Low'
    elif sa > 49:     return 'High'
    else:             return None

metadata['SA_status'] = metadata['sa_abundance'].apply(assign_sa_status)

# 3. NOW print the results
print(metadata[['sa_abundance', 'SA_status']].sort_values('sa_abundance'))


                       sa_abundance SA_status
NMR_sample_id                                
GYECZ1197.BACK.LES          0.00000       Low
GYECZ1197.RACF.NONLES       2.54032       Low
GYECZ1191.ABDO.LES         49.32039      High
GYECZ1193.LACF.NONLES      57.30307      High
GYECZ1191.ABDO.NONLES      67.47226      High
GYECZ1193.LACF.LES         89.31229      High


### 2.2: Clean Confidence Files

In [7]:
conf_paths  = conf_paths.dropna(subset=['feature'])
conf_metabs = conf_metabs.dropna(subset=['feature'])
 
conf_paths['feature_clean'] = (conf_paths['feature']
    .str.replace('.', ' ', regex=False)
    .str.replace('  ', ' ')
    .str.strip())
 
print("High-confidence pathways:")
print(conf_paths[['feature', 'Estimate', 'rho', 'p_value']].to_string())

High-confidence pathways:
                                                                  feature  Estimate       rho   p_value
0                          ARGININE.SYN4.PWY..L.ornithine.biosynthesis.II -0.000091 -0.569615  0.027795
1   ASPASN.PWY..superpathway.of.L.aspartate.and.L.asparagine.biosynthesis -0.000103 -0.635973  0.026376
2         GLYCOGENSYNTH.PWY..glycogen.biosynthesis.I..from.ADP.D.Glucose. -0.000105 -0.882601  0.000294
3                 HEME.BIOSYNTHESIS.II.1..heme.b.biosynthesis.V..aerobic. -0.000084 -0.636338  0.014089
4                             P161.PWY..acetylene.degradation..anaerobic. -0.000071 -0.555333  0.020157
5                 PWY.1269..CMP.3.deoxy.D.manno.octulosonate.biosynthesis -0.000047 -0.658579  0.015129
6             PWY.5136..fatty.acid..beta..oxidation.II..plant.peroxisome. -0.000112 -0.601741  0.043558
7           PWY.6292..superpathway.of.L.cysteine.biosynthesis..mammalian. -0.000110 -0.643751  0.025649
8                                    P

### 2.3: Compute High/Low S. aureus Group Means for Pathways 

In [8]:
# force numeric before taking means

model_samples = metadata[metadata['SA_status'].isin(['High', 'Low'])].copy()

high_ids = model_samples[model_samples['SA_status'] == 'High'].index.values
low_ids  = model_samples[model_samples['SA_status'] == 'Low'].index.values

print(f"High SA: {len(high_ids)} samples")
print(f"Low SA:  {len(low_ids)} samples")

# Force numeric — drops any stray text columns silently
pathways_numeric = pathways_from_master.apply(pd.to_numeric, errors='coerce')

# Check how many columns survived
n_before = pathways_from_master.shape[1]
n_after  = pathways_numeric.notna().any().sum()
print(f"\nPathway columns: {n_before} total, {n_after} numeric")

# Spot-check what got dropped (these are the stray metadata columns)
dropped = pathways_from_master.columns[pathways_numeric.isna().all()].tolist()
if dropped:
    print(f"Non-numeric columns dropped: {dropped}")

high_pathways_mean = pathways_numeric.loc[high_ids].mean()
low_pathways_mean  = pathways_numeric.loc[low_ids].mean()

print(f"\nHigh SA pathway mean range: {high_pathways_mean.min():.4f} to {high_pathways_mean.max():.4f}")
print(f"Low SA pathway mean range:  {low_pathways_mean.min():.4f} to {low_pathways_mean.max():.4f}")
 

High SA: 4 samples
Low SA:  2 samples

Pathway columns: 248 total, 248 numeric

High SA pathway mean range: 0.0000 to 0.0007
Low SA pathway mean range:  0.0000 to 0.0004


In [9]:
# Check the status of all 6 of your confirmed samples
print(metadata.loc[common_ids, ['sa_abundance', 'SA_status']])

                       sa_abundance SA_status
NMR_sample_id                                
GYECZ1191.ABDO.LES         49.32039      High
GYECZ1191.ABDO.NONLES      67.47226      High
GYECZ1193.LACF.LES         89.31229      High
GYECZ1193.LACF.NONLES      57.30307      High
GYECZ1197.BACK.LES          0.00000       Low
GYECZ1197.RACF.NONLES       2.54032       Low


## 3: Verification

In [10]:
print("=" * 60)
print("FULL DATA VERIFICATION")
print("=" * 60)
 
checks = {
    "Metadata":    metadata,
    "Pathways":    pathways_from_master,
    "NMR":         nmr,
    "Microbiome":  microbiome,
}
for name, df in checks.items():
    print(f"\n[{name}]")
    print(f"  Shape:     {df.shape}")
    print(f"  Any NaNs:  {df.isna().sum().sum()}")
    print(f"  Index[:3]: {df.index[:3].tolist()}")
 
print(f"\n[Single-cell expression]")
for name, df in [("mean_expr", mean_expr),
                 ("mean_expr_high", mean_expr_high),
                 ("mean_expr_low", mean_expr_low)]:
    print(f"\n  {name}:")
    print(f"    Shape:       {df.shape}  (expect 23017 × 11)")
    print(f"    Value range: {df.values.min():.3f} to {df.values.max():.3f}  (expect 0–3.5)")
    print(f"    Any NaNs:    {df.isna().sum().sum()}")
 
# Shared genes and cell types across expression matrices
shared_genes = (mean_expr.index
                .intersection(mean_expr_high.index)
                .intersection(mean_expr_low.index))
shared_cts   = sorted(set(mean_expr.columns)
                      .intersection(mean_expr_high.columns)
                      .intersection(mean_expr_low.columns))
print(f"\n  Shared genes:      {len(shared_genes):,}")
print(f"  Shared cell types: {shared_cts}")
 
print(f"\n{'=' * 60}")
print("READY FOR GEM" if len(shared_genes) > 0 else "⚠ CHECK ABOVE BEFORE PROCEEDING")
print("=" * 60)

FULL DATA VERIFICATION

[Metadata]
  Shape:     (6, 18)
  Any NaNs:  0
  Index[:3]: ['GYECZ1191.ABDO.LES', 'GYECZ1191.ABDO.NONLES', 'GYECZ1193.LACF.LES']

[Pathways]
  Shape:     (6, 248)
  Any NaNs:  0
  Index[:3]: ['GYECZ1191.ABDO.LES', 'GYECZ1191.ABDO.NONLES', 'GYECZ1193.LACF.LES']

[NMR]
  Shape:     (6, 38)
  Any NaNs:  0
  Index[:3]: ['GYECZ1191.ABDO.LES', 'GYECZ1191.ABDO.NONLES', 'GYECZ1193.LACF.LES']

[Microbiome]
  Shape:     (6, 44)
  Any NaNs:  0
  Index[:3]: ['GYECZ1191.ABDO.LES', 'GYECZ1191.ABDO.NONLES', 'GYECZ1193.LACF.LES']

[Single-cell expression]

  mean_expr:
    Shape:       (23017, 11)  (expect 23017 × 11)
    Value range: 0.000 to 5.963  (expect 0–3.5)
    Any NaNs:    0

  mean_expr_high:
    Shape:       (23017, 11)  (expect 23017 × 11)
    Value range: 0.000 to 6.130  (expect 0–3.5)
    Any NaNs:    0

  mean_expr_low:
    Shape:       (23017, 11)  (expect 23017 × 11)
    Value range: 0.000 to 6.089  (expect 0–3.5)
    Any NaNs:    0

  Shared genes:      23,01

In [11]:
print(mean_expr.columns.tolist())

['IFE SB', 'OB', 'IFE B', 'KER2', 'T CELL', 'IFE C', 'KER1', 'MEL', 'B CELL', 'uHF SB', 'FIB']


## 4: Get Target Metabolite IDs that Match Human2 GEM Style

In [12]:
# 1. Load your reference file
mets = pd.read_csv("/Users/maisievarcoe/Desktop/Research_Project/Scripts/Integration/GEM/pyglpk/Data/metabolites.tsv", sep="\t")

# 2. Define the complete mapping of Metabolite Name to KEGG ID
# This dictionary contains all the metabolites you identified
kegg_map = {
    "Succinic acid": "C00042", "Lactic acid": "C00186", "Proline": "C00148",
    "Lysine": "C00047", "Alanine": "C00041", "Tyrosine": "C00082",
    "Isoleucine": "C00407", "Tryptophan": "C00078", "Glucose": "C00031",
    "Arginine": "C00062", "Acetic acid": "C00033", "Asparagine": "C00152",
    "Aspartic acid": "C00049", "Choline": "C00114", "Citrulline": "C00327",
    "Formic acid": "C00058", "Fumaric acid": "C00122", "Glutamic acid": "C00025",
    "Glycerol": "C00116", "Histidine": "C00105", "Leucine": "C00123",
    "Maleic acid": "C00523", "Ornithine": "C00077", "Phenylalanine": "C00079",
    "Pyroglutamic acid": "C00625", "Serine": "C00065", "Threonine": "C00188",
    "Urocanic acid": "C00938", "Valine": "C00183"
}

# 3. Build the map that links Name → MAM ID
metabolite_map = {}

for name, kegg_id in kegg_map.items():
    # Use the KEGG ID as the anchor to find the row in your 'mets' file
    match = mets[mets['metKEGGID'] == kegg_id]
    
    if not match.empty:
        # Grab the 'mets' column value (e.g., 'MAM02943c')
        mam_id = match['mets'].values[0]
        metabolite_map[name] = mam_id
        print(f"Mapped: {name} → {mam_id}")
    else:
        print(f"Skipped: {name} (KEGG ID {kegg_id} not found in model)")

# 4. Finalize the dictionary for your gem_data
all_gem_metabolites = list(metabolite_map.keys())

# Update your gem_data object
gem_data["metabolite_map"] = metabolite_map
gem_data["gem_metabolites"] = all_gem_metabolites

print(f"\nSuccessfully mapped {len(metabolite_map)} metabolites.")

Mapped: Succinic acid → MAM02943c
Mapped: Lactic acid → MAM02403c
Mapped: Proline → MAM02770c
Mapped: Lysine → MAM02426c
Mapped: Alanine → MAM01307c
Mapped: Tyrosine → MAM03101c
Mapped: Isoleucine → MAM02184c
Mapped: Tryptophan → MAM03089c
Mapped: Glucose → MAM01965c
Mapped: Arginine → MAM01365c
Mapped: Acetic acid → MAM01252c
Mapped: Asparagine → MAM01369c
Mapped: Aspartic acid → MAM01370c
Mapped: Choline → MAM01513c
Mapped: Citrulline → MAM01588c
Mapped: Formic acid → MAM01833c
Mapped: Fumaric acid → MAM01862c
Mapped: Glutamic acid → MAM01974c
Mapped: Glycerol → MAM01983c
Mapped: Histidine → MAM03114c
Mapped: Leucine → MAM02360c
Mapped: Maleic acid → MAM01338c
Mapped: Ornithine → MAM02658c
Mapped: Phenylalanine → MAM02724c
Skipped: Pyroglutamic acid (KEGG ID C00625 not found in model)
Mapped: Serine → MAM02896c
Mapped: Threonine → MAM02993c
Skipped: Urocanic acid (KEGG ID C00938 not found in model)
Mapped: Valine → MAM03135c


NameError: name 'gem_data' is not defined

## 5: Package and Save all GEM-Ready Data

In [13]:
# Ensure metadata and pathways are restricted to the 6 shared samples
metadata = metadata.loc[common_ids]
pathways_from_master = pathways_from_master.loc[common_ids]

# 1. Map your metabolites using the KEGG bridge (The 27 that worked)
# Note: Ensure kegg_map is defined as in the previous step
metabolite_map = {}
for name, kegg_id in kegg_map.items():
    match = mets[mets['metKEGGID'] == kegg_id]
    if not match.empty:
        metabolite_map[name] = match['mets'].values[0]
        print(f"Mapped: {name} → {metabolite_map[name]}")
    else:
        print(f"Skipped: {name}")

# 2. Update the gem_data dictionary
gem_data = {
    "metadata": metadata,
    "nmr": nmr,
    "microbiome": microbiome,
    "pathways": pathways_from_master,
    "mean_expr": mean_expr,
    "mean_expr_high": mean_expr_high,
    "mean_expr_low": mean_expr_low,
    "metabolite_map": metabolite_map,
    "gem_metabolites": list(metabolite_map.keys()), # Now contains all 27 mapped names
    "common_ids": common_ids
}

gem_data["mean_expr_by_sample"] = {
    "KER1":   mean_expr_ker1_by_sample,
    "T CELL": mean_expr_tcell_by_sample,
}

gem_data["cell_counts_by_sample"] = {
    "KER1":   cell_counts_ker1,
    "T CELL": cell_counts_tcell,
}
# 3. Save the final pkl file
OUTPUT_PATH = f"{BASE}/gem_input_data7.pkl"
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(gem_data, f)

# 4. Final verification output
print(f"\nSuccessfully saved updated GEM data to: {OUTPUT_PATH}")
print(f"Total metabolites mapped: {len(metabolite_map)}")
print("Contents:")
for k, v in gem_data.items():
    if hasattr(v, 'shape'):
        print(f"  {k:20s} → shape {v.shape}")
    else:
        print(f"  {k:20s} → {type(v)}")

Mapped: Succinic acid → MAM02943c
Mapped: Lactic acid → MAM02403c
Mapped: Proline → MAM02770c
Mapped: Lysine → MAM02426c
Mapped: Alanine → MAM01307c
Mapped: Tyrosine → MAM03101c
Mapped: Isoleucine → MAM02184c
Mapped: Tryptophan → MAM03089c
Mapped: Glucose → MAM01965c
Mapped: Arginine → MAM01365c
Mapped: Acetic acid → MAM01252c
Mapped: Asparagine → MAM01369c
Mapped: Aspartic acid → MAM01370c
Mapped: Choline → MAM01513c
Mapped: Citrulline → MAM01588c
Mapped: Formic acid → MAM01833c
Mapped: Fumaric acid → MAM01862c
Mapped: Glutamic acid → MAM01974c
Mapped: Glycerol → MAM01983c
Mapped: Histidine → MAM03114c
Mapped: Leucine → MAM02360c
Mapped: Maleic acid → MAM01338c
Mapped: Ornithine → MAM02658c
Mapped: Phenylalanine → MAM02724c
Skipped: Pyroglutamic acid
Mapped: Serine → MAM02896c
Mapped: Threonine → MAM02993c
Skipped: Urocanic acid
Mapped: Valine → MAM03135c

Successfully saved updated GEM data to: /Users/maisievarcoe/Desktop/Research_Project/Scripts/Integration/GEM/pyglpk/Data/gem_input

### Final Validation to make sure i have the correct data

In [14]:
# ============================================================
# 6: FINAL CHECK — EXACT DATA AVAILABLE FOR EACH MODALITY
# ============================================================

print("\n" + "="*70)
print("FINAL GEM INPUT CHECK")
print("="*70)

# ------------------------------------------------------------
# 1. SAMPLE ALIGNMENT
# ------------------------------------------------------------
print("\n[1] SAMPLE ALIGNMENT")
print(f"  Shared sample IDs (n={len(common_ids)}):")
print(" ", common_ids)

for name, df in {
    "metadata": metadata,
    "nmr": nmr,
    "microbiome": microbiome,
    "pathways": pathways_from_master
}.items():
    print(f"  {name:12s} → shape {df.shape}, index matches shared: {set(df.index)==set(common_ids)}")

# ------------------------------------------------------------
# 2. MODALITY CONTENT SUMMARY
# ------------------------------------------------------------
print("\n[2] MODALITY CONTENT SUMMARY")

def modality_summary(name, df):
    print(f"\n{name}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Missing values: {df.isna().sum().sum()}")
    if df.shape[1] < 20:
        print(f"  Column names: {list(df.columns)}")
    else:
        print(f"  First 5 columns: {list(df.columns[:5])}")

modality_summary("Metadata", metadata)
modality_summary("NMR metabolites", nmr)
modality_summary("Microbiome species", microbiome)
modality_summary("Pathways", pathways_from_master)

# ------------------------------------------------------------
# 3. SINGLE-CELL EXPRESSION SUMMARY
# ------------------------------------------------------------
print("\n[3] SINGLE-CELL EXPRESSION")

for name, df in {
    "mean_expr": mean_expr,
    "mean_expr_high": mean_expr_high,
    "mean_expr_low": mean_expr_low
}.items():
    print(f"\n{name}")
    print(f"  Shape: {df.shape}")
    print(f"  Genes: {df.shape[0]}, Cell types: {df.shape[1]}")
    print(f"  Value range: {df.values.min():.3f} to {df.values.max():.3f}")
    print(f"  Missing values: {df.isna().sum().sum()}")

# ------------------------------------------------------------
# 4. METABOLITE MAPPING CHECK
# ------------------------------------------------------------
print("\n[4] METABOLITE MAPPING CHECK")

print("  Metabolite:")
for m in gem_data["gem_metabolites"]:
    print("   -", m)

print("\n  Human-GEM IDs found:")
for k, v in metabolite_map.items():
    print(f"   {k:15s} → {v}")

missing = [m for m in gem_data["gem_metabolites"] if m not in metabolite_map]
if missing:
    print("\n  ⚠ Missing mappings:", missing)
else:
    print("\n  ✓ All metabolites successfully mapped to Human-GEM IDs")

# ------------------------------------------------------------
# 5. CHECK THAT ALL 10 METABOLITES EXIST IN THE NMR MATRIX
# ------------------------------------------------------------
print("\n[5] CHECK NMR MATRIX FOR TARGET METABOLITES")

nmr_missing = [m for m in gem_data["gem_metabolites"] if m not in nmr.columns]
if nmr_missing:
    print("  ⚠ Missing from NMR:", nmr_missing)
else:
    print("  ✓ All metabolites present in NMR data")

# ------------------------------------------------------------
# 6. FINAL READINESS
# ------------------------------------------------------------
print(f"Current common_ids length: {len(common_ids)}")
print("\n" + "="*70)
print("GEM INPUT STATUS")
print("="*70)

if (
    len(common_ids) == 6
    and not missing
    and not nmr_missing
):
    print("✓ All modalities aligned")
    print("✓ All metabolites mapped")
    print("✓ All metabolites present in NMR")
    print("✓ Data is READY for GEM modelling")
else:
    print("⚠ Something needs fixing before GEM modelling")



FINAL GEM INPUT CHECK

[1] SAMPLE ALIGNMENT
  Shared sample IDs (n=6):
  ['GYECZ1191.ABDO.LES', 'GYECZ1191.ABDO.NONLES', 'GYECZ1193.LACF.LES', 'GYECZ1193.LACF.NONLES', 'GYECZ1197.BACK.LES', 'GYECZ1197.RACF.NONLES']
  metadata     → shape (6, 18), index matches shared: True
  nmr          → shape (6, 38), index matches shared: True
  microbiome   → shape (6, 44), index matches shared: True
  pathways     → shape (6, 248), index matches shared: True

[2] MODALITY CONTENT SUMMARY

Metadata
  Shape: (6, 18)
  Columns: 18
  Missing values: 0
  Column names: ['eg_id', 'sample_short', 'sample_id', 'sa_abundance', 'sample_number', 'sample_site', 'lesional', 'local_EASI_intensity', 's_aureus_culture_growth_viapath', 'age', 'sex', 'EASI', 'weeks_since_last_systemic', 'weeks_since_last_TCS_or_TCI', 'days_since_any_topicals', 'TEWL_mean', 'pH_mean', 'SA_status']

NMR metabolites
  Shape: (6, 38)
  Columns: 38
  Missing values: 0
  First 5 columns: ['Leucine', 'Isoleucine', 'Valine', 'Unknown 1', 

In [15]:
print("Keys being saved:", list(gem_data.keys()))
for ct in ["KER1", "T CELL"]:
    print(f"  mean_expr_by_sample[{ct}]: {gem_data['mean_expr_by_sample'][ct].shape}")
    print(f"  cell_counts_by_sample[{ct}]: {gem_data['cell_counts_by_sample'][ct].shape}")

Keys being saved: ['metadata', 'nmr', 'microbiome', 'pathways', 'mean_expr', 'mean_expr_high', 'mean_expr_low', 'metabolite_map', 'gem_metabolites', 'common_ids', 'mean_expr_by_sample', 'cell_counts_by_sample']
  mean_expr_by_sample[KER1]: (23017, 6)
  cell_counts_by_sample[KER1]: (6, 1)
  mean_expr_by_sample[T CELL]: (23017, 6)
  cell_counts_by_sample[T CELL]: (6, 1)
